In [ ]:
# ================================
# 1. Import libraries
# ================================
import pandas as pd

# ================================
# 2. Load dataset
# ================================
df = pd.read_csv("cms_hospital_patient_satisfaction_2016.csv")

print("Original shape:", df.shape)

# ================================
# 3. Select useful columns
# ================================
df_clean = df[[
    'Facility Name',
    'State',
    'Hospital Type',
    'Emergency Services',
    'Hospital overall rating',
    'Patient Survey Star Rating',
    'HCAHPS Answer Percent',
    'HCAHPS Question'
]].copy()

# ================================
# 4. Replace invalid values
# ================================
df_clean = df_clean.replace(['Not Available', 'Not Applicable'], pd.NA)

# ================================
# 5. Convert to numeric
# ================================
numeric_cols = [
    'Hospital overall rating',
    'Patient Survey Star Rating',
    'HCAHPS Answer Percent'
]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# ================================
# 6. Drop rows with missing key values
# ================================
df_clean = df_clean.dropna(subset=[
    'Hospital overall rating',
    'HCAHPS Answer Percent'
])

# (Optional) Keep Patient Survey Rating if available
# df_clean = df_clean.dropna(subset=['Patient Survey Star Rating'])

# ================================
# 7. Remove unrealistic values
# ================================
df_clean = df_clean[
    (df_clean['Hospital overall rating'] >= 1) &
    (df_clean['Hospital overall rating'] <= 5) &
    (df_clean['HCAHPS Answer Percent'] >= 0) &
    (df_clean['HCAHPS Answer Percent'] <= 100)
]

# ================================
# 8. Create additional KPI columns
# ================================

# Satisfaction category
df_clean['Satisfaction Level'] = pd.cut(
    df_clean['Hospital overall rating'],
    bins=[0, 2, 4, 5],
    labels=['Low', 'Medium', 'High']
)

# ================================
# 9. Final check
# ================================
print("Cleaned shape:", df_clean.shape)
print(df_clean.describe())

# ================================
# 10. Export cleaned dataset
# ================================
df_clean.to_csv("cleaned_hospital_data.csv", index=False)

print("✅ Cleaned dataset saved as 'cleaned_hospital_data.csv'")

In [ ]:
# Create satisfaction category
df_clean['Satisfaction Level'] = pd.cut(
    df_clean['Hospital overall rating'],
    bins=[0, 2, 4, 5],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
from scipy.stats import chi2_contingency

# Contingency table
table = pd.crosstab(df_clean['Hospital Type'], df_clean['Satisfaction Level'])

# Chi-square test
chi2, p, dof, expected = chi2_contingency(table)

print("Chi2:", chi2)
print("P-value:", p)